In [1]:
!pip install biopython pandas numpy scikit-learn

import pandas as pd
import numpy as np
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio import SeqIO
from sklearn.ensemble import RandomForestClassifier
from collections import Counter
import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.9 MB/s eta 0:00:00


In [2]:
import random

# ================================
# Constants and Configurations
# ================================

AAS = "ACDEFGHIKLMNPQRSTVWY"
SW_WINDOWS = [20, 40]

AA_GROUPS = {
    "KR": "KR",
    "KRH": "KRH",
    "ED": "ED",
}

AA_GROUPS_EXT = {
    "STNQCH": "STNQCH",
    "ILMV":   "ILMV",
    "FWY":    "FWY",
}

AAC_GROUPS_ALL = {**AA_GROUPS_EXT, **AA_GROUPS}

def clean_seq(seq):
    """Replaces non-standard amino acid U (Selenocysteine) with C (Cysteine)."""
    return seq.replace("U", "C")

# ================================
# Data Loading & Preprocessing (High / Low)
# ================================

df_high = pd.read_excel("High_group_ID.xlsx")
df_low  = pd.read_excel("Low_group_ID.xlsx")

df_high["Class"] = 1
df_low["Class"]  = 0

df_class = pd.concat([df_high, df_low], ignore_index=True)

seqs = {
    rec.id: str(rec.seq)
    for rec in SeqIO.parse("ALL_protein_sequences.fasta", "fasta")
}

df_seq = pd.DataFrame(seqs.items(), columns=["ID", "Sequence"])

df = df_class.merge(df_seq, on="ID")
df["Sequence"] = df["Sequence"].map(clean_seq)

# ================================
# Feature Calculation Functions
# ================================

def termial_bias(seq, aa_set):
    """Calculates the mean relative positional bias from the center for a set of amino acids."""
    L = len(seq)
    pos = [abs((i + 0.5)/L - 0.5) for i, a in enumerate(seq) if a in aa_set]
    return np.mean(pos) if pos else 0.0

def sliding_window_var_norm(seq, groups):
    """Calculates normalized variance of group frequencies across sliding windows."""
    out = []
    L = len(seq)
    cnt = Counter(seq)

    for w in SW_WINDOWS:
        for g in groups:
            chars = g
            p = sum(cnt[a] for a in chars) / L

            vals = [
                sum(seq[i:i+w].count(a) for a in chars) / w
                for i in range(L - w + 1)
            ]

            var_obs = np.var(vals) if vals else 0.0
            var_exp = p * (1 - p) / w
            out.append(var_obs / (var_exp + 1e-6))
    return out

KMER2_SYM_LIST = sorted(
    set("".join(sorted(a + b)) for a in AAS for b in AAS)
)

def kmer2_features(seq):
    """Extracts normalized frequencies of symmetric 2-mers."""
    L = len(seq)
    cnt = Counter("".join(sorted(seq[i:i+2])) for i in range(L - 1))
    denom = max(L - 1, 1)
    return [cnt.get(k, 0) / denom for k in KMER2_SYM_LIST]

# ================================
# Feature Extraction Pipeline
# ================================

def extract_features(seq):
    prot = ProteinAnalysis(seq)
    aa = prot.count_amino_acids()
    L = len(seq)

    feat = []

    # --- Physicochemical properties ---
    feat += [
        prot.molecular_weight(),
        prot.gravy(),
        prot.charge_at_pH(7.5),
        prot.isoelectric_point()
    ]

    # --- Amino Acid Composition (AAC) ---
    feat += [aa[a] / L for a in AAS]

    # --- Grouped AAC ---
    feat += [
        sum(aa[a] for a in g) / L
        for g in AAC_GROUPS_ALL.values()
    ]

    # --- Terminal positional bias ---
    feat += [termial_bias(seq, a) for a in AAS]

    # --- Sliding window variance ---
    feat += sliding_window_var_norm(seq, AAS)
    feat += sliding_window_var_norm(seq, AA_GROUPS_EXT.values())
    feat += sliding_window_var_norm(seq, AA_GROUPS.values())

    # --- Symmetric 2-mer frequencies ---
    feat += kmer2_features(seq)

    return np.array(feat)

# ================================
# Model Training (Initial All Features)
# ================================

X = np.vstack([extract_features(s) for s in df["Sequence"]])
y = df["Class"].values

clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf.fit(X, y)

# ================================
# Feature Selection: Top 100 Features
# ================================

importances = clf.feature_importances_
TOP_FEATURE_IDX = np.argsort(importances)[-100:]

X_top = X[:, TOP_FEATURE_IDX]

clf_top = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf_top.fit(X_top, y)

print("Top features selected:", len(TOP_FEATURE_IDX))

# ================================
# Scoring Functions
# ================================

def predict_scores(seq):
    """Returns predicted probability for (Low, High) classes."""
    x_all = extract_features(seq)
    x = x_all[TOP_FEATURE_IDX].reshape(1, -1)

    prob = clf_top.predict_proba(x)[0]
    return prob[0], prob[1]   # Low, High

def score_diff(seq):
    """Calculates objective score to maximize Low class preference: Score = P(Low) - P(High)."""
    low, high = predict_scores(seq)
    return low - high

# ================================
# Mutation & Optimization Algorithm
# ================================

AAS_LIST = list(AAS)

MAX_ITER = 1000
MAX_MUT = 30

def optimize(seq):
    """Performs greedy single-point mutation optimization to increase score_diff."""
    seq = list(seq)

    best_seq = "".join(seq)
    best_score = score_diff(best_seq)
    mut_count = 0

    for _ in range(MAX_ITER):

        # Select random position
        i = random.randint(0, len(seq) - 1)
        orig = seq[i]

        # Try amino acids in random order
        for aa in random.sample(AAS_LIST, len(AAS_LIST)):
            if aa == orig:
                continue

            seq[i] = aa
            new = "".join(seq)
            s = score_diff(new)

            if s > best_score:
                best_seq = new
                best_score = s
                mut_count += 1
                break
            else:
                seq[i] = orig

        if mut_count >= MAX_MUT:
            break

    return best_seq, best_score, mut_count

# ================================
# Execution Parameters
# ================================

N_CANDIDATES = 1

# ================================
# FASTA Processing & Output
# ================================

INPUT_FASTA  = "Designed_High_seq.fasta"
OUTPUT_FASTA = "Mutated_Designed_High_seq.fasta"

def process_fasta():
    results = []

    for rec in SeqIO.parse(INPUT_FASTA, "fasta"):
        print("Processing:", rec.id)

        seq = str(rec.seq)

        # --- Run optimization N times per input sequence ---
        for i in range(N_CANDIDATES):

            mut_seq, score, n = optimize(seq)
            low, high = predict_scores(mut_seq)

            results.append((
                f"{rec.id}_cand{i+1}",
                mut_seq,
                score,
                n,
                low,
                high
            ))

    # --- Export results to FASTA file ---
    with open(OUTPUT_FASTA, "w") as f:
        for id_, seq, score, n, low, high in results:
            f.write(
                f">{id_}|mut={n}|score={score:.4f}|low={low:.4f}|high={high:.4f}\n"
            )
            f.write(seq + "\n")

process_fasta()

Top features selected: 100
Processing: seq_000|score=0.933
